# 03 . nn.Module

## Concept — nn.Module (revision notes)

**What it is**
- `nn.Module` is the base class for every layer and every model in PyTorch.
- A single `nn.Linear` layer is an `nn.Module`. A 100-layer transformer is
  also an `nn.Module` — usually built by nesting many smaller `nn.Module`s
  inside each other.
- Learn this class well and you can read (and write) any PyTorch model, from
  a two-line MLP to a HuggingFace transformer.

**Why it exists**
- A model needs to: track its learnable weights, know how to move them to a
  GPU, know how to save/load them, and know how to run a forward pass —
  consistently, regardless of what the model actually computes.
- `nn.Module` gives all of this for free once you follow its conventions.

**The two required pieces of every subclass**
- `__init__(self, ...)` — declare the layers/parameters you'll use.
  - **Must** call `super().__init__()` first, or nothing below it works.
- `forward(self, x)` — defines what happens to input `x`.
  - You never call `.forward()` directly — call the module itself,
    `model(x)`, which runs bookkeeping (hooks, etc.) and then calls
    `forward` for you internally.

**Parameters — the learnable tensors**
- Wrapped in `nn.Parameter(...)`, or come automatically from sub-layers like
  `nn.Linear`.
- Automatically registered and appear in `model.parameters()` — exactly what
  you hand to an optimizer: `optimizer = torch.optim.Adam(model.parameters())`.
- A plain `self.x = torch.randn(3)` in `__init__` is **invisible** to
  `.parameters()`, `.to(device)`, and `state_dict()` — a common silent bug.

**Buffers — model state that is NOT learned**
- Registered with `self.register_buffer("name", tensor)`.
- Example: the running mean/variance inside `BatchNorm`.
- Buffers move with `.to(device)` and get saved/loaded with the model, but
  the optimizer **never** updates them via gradient descent.

**state_dict() — the model's persistent state**
- An ordered dict of every parameter and buffer, keyed by name.
- This is what you save to disk and load back — the entire persistent state
  of the model, decoupled from the Python class that produced it.
- Best practice: save `model.state_dict()`, not the whole `model` object
  (saving the whole object pickles the class definition too, which is
  fragile across refactors/versions).

**Composition — how real models are built**
- `nn.Sequential` — for simple straight-line stacks of layers, no branching.
- `nn.ModuleList` — like a Python list, but registers every module properly
  so parameters/buffers/device-moves all work.
- `nn.ModuleDict` — like a Python dict, same registration guarantee.
- **The trap:** a plain Python `list` or `dict` of modules looks like it
  works (forward pass runs fine) but silently registers **zero** parameters
  — `.parameters()`, `.to(device)`, and `state_dict()` will all miss them.

**train() vs eval() mode**
- `model.train()` — default mode; layers like `Dropout` and `BatchNorm`
  behave in their "training" way (dropout active, batch statistics used).
- `model.eval()` — switches those same layers to inference behavior
  (dropout disabled, running statistics used instead of batch statistics).
- Forgetting `model.eval()` before evaluation is one of the most common
  real-world bugs — it silently makes eval metrics noisy or wrong.
- Always pair with `torch.no_grad()` during evaluation for memory/speed too.

**Useful inspection methods, quick reference**
- `model.named_parameters()` — `(name, tensor)` pairs, most useful for debugging.
- `model.named_buffers()` — same, for buffers.
- `model.named_modules()` — every submodule by name, including itself.
- `model.state_dict()` — full persistent state as an OrderedDict.
- `sum(p.numel() for p in model.parameters())` — total parameter count.
- `model.zero_grad()` — zeroes gradients of all parameters (same effect as
  `optimizer.zero_grad()` when the optimizer covers all of them).


### 1. A minimal custom module

**What this cell does (plain language):** we define the smallest possible
custom model — one linear layer plus a ReLU — by subclassing `nn.Module`.
We create an instance, run a batch of random input through it, and print
the model and its output shape so you can see the whole lifecycle: define,
instantiate, call.


In [31]:
import torch
import torch.nn as nn

class TinyNet(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()  # MUST be called first - registers the module machinery
        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.linear(x)
        x = self.activation(x)
        return x

model = TinyNet(4, 2)
x = torch.randn(3, 4)   # batch of 3 samples, 4 features each
out = model(x)          # call the MODULE, not model.forward(x) directly
print("output shape:", out.shape)
print(model)  # nn.Module has a nice default string representation


output shape: torch.Size([3, 2])
TinyNet(
  (linear): Linear(in_features=4, out_features=2, bias=True)
  (activation): ReLU()
)


### 2. Parameters — what the optimizer actually updates

**What this cell does (plain language):** we loop over every learnable
parameter in the model, printing its name, shape, and whether it requires
gradients — this is exactly what you'd do to sanity-check a model before
training, or to debug "why isn't this layer training."


In [32]:
for name, param in model.named_parameters():
    print(name, "| shape:", tuple(param.shape), "| requires_grad:", param.requires_grad)

total_params = sum(p.numel() for p in model.parameters())
print("\ntotal learnable parameters:", total_params)
# Linear(4, 2) has: weight (2x4 = 8) + bias (2) = 10 total, matches above


linear.weight | shape: (2, 4) | requires_grad: True
linear.bias | shape: (2,) | requires_grad: True

total learnable parameters: 10


### 3. Buffers — model state that ISN'T learned

**What this cell does (plain language):** we build a tiny module that tracks
a running average using `register_buffer`, feed it a few batches, and then
prove that the buffer shows up in `state_dict()` (so it's saved/loaded and
moved with the model) but does NOT show up in `.parameters()` (so the
optimizer will never try to update it with gradients).


In [33]:
class RunningAverage(nn.Module):
    def __init__(self):
        super().__init__()
        # register_buffer: tracked, moves with .to(device), saved in
        # state_dict, but NEVER in .parameters() and NEVER updated by the optimizer
        self.register_buffer("running_mean", torch.zeros(1))
        self.count = 0  # a plain Python int - fine, doesn't need registering since it's not a tensor

    def update(self, x):
        self.count += 1
        self.running_mean += (x.mean() - self.running_mean) / self.count

tracker = RunningAverage()
for batch in [torch.randn(10) + i for i in range(3)]:
    tracker.update(batch)

print("running_mean buffer:", tracker.running_mean)
print("is it in .parameters()?", list(tracker.parameters()), " <- empty, correct!")
print("is it in .state_dict()?", tracker.state_dict(), " <- present, correct!")


running_mean buffer: tensor([1.0914])
is it in .parameters()? []  <- empty, correct!
is it in .state_dict()? OrderedDict({'running_mean': tensor([1.0914])})  <- present, correct!


### 4. state_dict — saving and loading model weights

**What this cell does (plain language):** we save a model's weights to disk
using the recommended method (`state_dict()`, not the whole model object),
then load them into a *fresh* instance of the same architecture, and prove
both instances now produce identical output on the same input.


In [34]:
model = TinyNet(4, 2)

# Save just the weights (recommended) - NOT the whole model object
torch.save(model.state_dict(), "/tmp/tinynet_weights.pt")

# Load into a brand new instance of the SAME architecture
model2 = TinyNet(4, 2)
model2.load_state_dict(torch.load("/tmp/tinynet_weights.pt"))

# Verify they now produce identical output
x = torch.randn(2, 4)
with torch.no_grad():
    out1 = model(x)
    out2 = model2(x)
print("outputs match after load_state_dict:", torch.allclose(out1, out2))
print("\nstate_dict keys:", list(model.state_dict().keys()))


outputs match after load_state_dict: True

state_dict keys: ['linear.weight', 'linear.bias']


### 5. Composition: Sequential, ModuleList, and the plain-list trap

**What this cell does (plain language):** we build the same kind of
multi-layer model three ways — `nn.Sequential`, the correct `nn.ModuleList`
approach, and the WRONG plain-Python-list approach — and count how many
parameter tensors get registered in each case, so you can see the plain-list
bug produce zero registered parameters even though the model still "runs."


In [35]:
# nn.Sequential - simplest option, for straight-line stacks with no branching
seq_model = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 1),
)
print("Sequential params registered:", len(list(seq_model.parameters())), "tensors")

# nn.ModuleList - like a Python list, but registers every module correctly
class MultiLayer(nn.Module):
    def __init__(self, sizes):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(sizes[i], sizes[i+1]) for i in range(len(sizes)-1)])

    def forward(self, x):
        for layer in self.layers:
            x = torch.relu(layer(x))
        return x

good_model = MultiLayer([10, 20, 20, 1])
print("ModuleList version - params found:", len(list(good_model.parameters())), "tensors")

# THE TRAP: a plain Python list LOOKS like it works (forward pass runs fine)
# but silently registers NOTHING
class BadMultiLayer(nn.Module):
    def __init__(self, sizes):
        super().__init__()
        self.layers = [nn.Linear(sizes[i], sizes[i+1]) for i in range(len(sizes)-1)]  # plain list!

    def forward(self, x):
        for layer in self.layers:
            x = torch.relu(layer(x))
        return x

bad_model = BadMultiLayer([10, 20, 20, 1])
test_input = torch.randn(2, 10)
print("\nBadMultiLayer forward pass still 'works':", bad_model(test_input).shape)
print("but params found:", len(list(bad_model.parameters())), "tensors  <- BUG: should be 6!")
print("this means .to(device) and state_dict() will silently miss these weights entirely")


Sequential params registered: 4 tensors
ModuleList version - params found: 6 tensors

BadMultiLayer forward pass still 'works': torch.Size([2, 1])
but params found: 0 tensors  <- BUG: should be 6!
this means .to(device) and state_dict() will silently miss these weights entirely


### 6. train() vs eval() mode

**What this cell does (plain language):** we build a module with dropout,
run the exact same input through it in `train()` mode and then in `eval()`
mode, and print both outputs side by side so you can see dropout actually
randomly zero things out in train mode but do nothing in eval mode.


In [36]:
class WithDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        return self.dropout(x)

m = WithDropout()
x = torch.ones(10)

m.train()  # default mode - dropout is ACTIVE
train_out = m(x)
print("train mode output (dropout active, ~half zeroed randomly):", train_out)

m.eval()  # dropout becomes a no-op, as it should be at inference time
eval_out = m(x)
print("eval mode output (dropout disabled, unchanged input):", eval_out)

# Forgetting model.eval() before evaluation silently makes eval metrics
# noisy/wrong - always call it before running validation or inference.


train mode output (dropout active, ~half zeroed randomly): tensor([0., 0., 0., 2., 2., 2., 0., 2., 0., 2.])
eval mode output (dropout disabled, unchanged input): tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])


### 7. Custom parameters without a built-in layer

**What this cell does (plain language):** sometimes you need a learnable
weight that isn't just "a Linear layer" — here we build a tiny custom module
with a hand-created `nn.Parameter`, to show exactly how you'd wire up your
own learnable weight from scratch.


In [37]:
class ScaleAndShift(nn.Module):
    """Learns y = x * scale + shift, elementwise, with scale/shift as learnable scalars."""
    def __init__(self):
        super().__init__()
        # nn.Parameter wraps a tensor so it's automatically registered
        self.scale = nn.Parameter(torch.tensor(1.0))
        self.shift = nn.Parameter(torch.tensor(0.0))

    def forward(self, x):
        return x * self.scale + self.shift

m = ScaleAndShift()
print("registered params:", [(n, p.item()) for n, p in m.named_parameters()])

x = torch.tensor([1.0, 2.0, 3.0])
out = m(x)
print("output:", out)

# prove it's trainable - run one manual gradient step
loss = (out - torch.tensor([2.0, 4.0, 6.0])).pow(2).sum()
loss.backward()
print("\ngrad w.r.t. scale:", m.scale.grad.item())
print("grad w.r.t. shift:", m.shift.grad.item())


registered params: [('scale', 1.0), ('shift', 0.0)]
output: tensor([1., 2., 3.], grad_fn=<AddBackward0>)

grad w.r.t. scale: -28.0
grad w.r.t. shift: -12.0


### 8. Freezing parameters (requires_grad = False)

**What this cell does (plain language):** a very common production pattern
— take a pretrained model, freeze most of it, and only fine-tune a specific
layer. We simulate that here on a tiny two-layer model, freezing the first
layer and confirming only the second layer's parameters accumulate gradients.


In [38]:
class TwoLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 10)
        self.layer2 = nn.Linear(10, 1)

    def forward(self, x):
        return self.layer2(torch.relu(self.layer1(x)))

model = TwoLayer()

# Freeze layer1 - it will NOT be updated by the optimizer
for param in model.layer1.parameters():
    param.requires_grad = False

x = torch.randn(4, 10)
target = torch.randn(4, 1)
loss = ((model(x) - target) ** 2).mean()
loss.backward()

print("layer1.weight.requires_grad:", model.layer1.weight.requires_grad)
print("layer1.weight.grad is None:", model.layer1.weight.grad is None, "<- frozen, no grad computed")
print("layer2.weight.requires_grad:", model.layer2.weight.requires_grad)
print("layer2.weight.grad is None:", model.layer2.weight.grad is None, "<- trains normally")

# Best practice: only pass trainable params to the optimizer
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = torch.optim.Adam(trainable_params, lr=1e-3)
print("\noptimizer will update", sum(p.numel() for p in model.parameters() if p.requires_grad), "parameters")


layer1.weight.requires_grad: False
layer1.weight.grad is None: True <- frozen, no grad computed
layer2.weight.requires_grad: True
layer2.weight.grad is None: False <- trains normally

optimizer will update 11 parameters


### 9. Moving a model (and everything inside it) to a device

**What this cell does (plain language):** we build a model containing both
parameters and a buffer, move the whole thing with one `.to(device)` call,
and confirm both the parameters AND the buffer moved together — this is the
"consistent bookkeeping" that `nn.Module` provides for free.


In [39]:
class TinyWithBuffer(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("counter", torch.zeros(1))

model = TinyWithBuffer()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)  # moves EVERY registered parameter AND buffer, recursively

print("linear.weight device:", model.linear.weight.device)
print("counter buffer device:", model.counter.device)
print("both moved together with a single .to(device) call")

# NOTE: unlike tensors, model.to(device) IS effectively in-place for the
# module - you don't need to reassign `model = model.to(device)`, though
# doing so is harmless and a common defensive habit.


linear.weight device: cuda:0
counter buffer device: cuda:0
both moved together with a single .to(device) call


### 10. Inspecting a model's structure

**What this cell does (plain language):** for debugging a model you didn't
write yourself (e.g. from a library), these are the standard ways to see
what's inside it — printing the module tree, listing named submodules, and
counting parameters per submodule.


In [40]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3)
        self.fc = nn.Linear(16, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = x.mean(dim=[2, 3])  # global average pool
        return self.fc(x)

model = SmallCNN()
print("full model printout:")
print(model)

print("\nnamed_modules (includes the model itself as ''):")
for name, module in model.named_modules():
    print(f"  '{name}' -> {module.__class__.__name__}")

print("\nparameter count per top-level submodule:")
for name, module in model.named_children():
    count = sum(p.numel() for p in module.parameters())
    print(f"  {name}: {count} parameters")


full model printout:
SmallCNN(
  (conv1): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc): Linear(in_features=16, out_features=10, bias=True)
)

named_modules (includes the model itself as ''):
  '' -> SmallCNN
  'conv1' -> Conv2d
  'conv2' -> Conv2d
  'fc' -> Linear

parameter count per top-level submodule:
  conv1: 224 parameters
  conv2: 1168 parameters
  fc: 170 parameters


## Common bugs

- **Forgot `super().__init__()`**
  `AttributeError: cannot assign module before Module.__init__() call`.
  Fix: always call `super().__init__()` as the very first line of `__init__`.

- **Parameters missing from `.parameters()` / not moving to GPU**
  Usually caused by storing sub-modules in a plain Python `list`/`dict`
  instead of `nn.ModuleList`/`nn.ModuleDict`, or assigning a raw tensor
  instead of wrapping it in `nn.Parameter`. Fix: use the container classes,
  and wrap any hand-created learnable tensor in `nn.Parameter(...)`.

- **Forgot `model.eval()` before evaluation/inference**
  Dropout keeps randomly zeroing activations and BatchNorm keeps using
  batch-level statistics instead of running statistics — eval metrics become
  noisy or wrong. Fix: always call `model.eval()` before evaluation, and
  `model.train()` again before resuming training. Wrap eval code in
  `with torch.no_grad():` too, for memory/speed.

- **`RuntimeError: Error(s) in loading state_dict` — key mismatch**
  The saved `state_dict` doesn't match the current model's architecture
  exactly (different layer names, sizes, or an extra/missing layer). Fix:
  make sure the class definition matches exactly what saved the weights, or
  use `load_state_dict(sd, strict=False)` deliberately for a partial/adapted
  load (e.g. fine-tuning with new layers).

- **Saved the whole model object instead of `state_dict()`**
  `torch.save(model, path)` pickles the entire class definition along with
  the weights — fragile across code/refactors/versions. Fix: prefer
  `torch.save(model.state_dict(), path)` and reconstruct the architecture
  in code before loading weights back in.

- **Buffer not moving to GPU / not saved correctly**
  Forgot `register_buffer` and just assigned a raw tensor attribute instead.
  It won't move with `.to(device)` and won't appear in `state_dict()`. Fix:
  any persistent tensor that isn't a learnable parameter still needs
  `register_buffer` if it should travel with the model.

- **Froze the wrong parameters, or forgot to filter them for the optimizer**
  Setting `requires_grad = False` stops gradients from being computed, but
  if you still pass `model.parameters()` (unfiltered) to the optimizer, it's
  harmless but wasteful, and more importantly it's easy to freeze the wrong
  layer and not notice. Fix: after freezing, print
  `[(n, p.requires_grad) for n, p in model.named_parameters()]` to confirm
  exactly what's frozen before training.

- **Calling `model.forward(x)` directly instead of `model(x)`**
  Works most of the time but skips PyTorch's internal hooks (used by things
  like forward hooks for debugging/visualization). Fix: always call the
  module itself, `model(x)`, never `.forward()` directly.


## Exercises

Attempt each one in the empty cell right after the question. My full solution is in the cell after that - don't peek until you've tried.

**Exercise 1 - Spot the bug.**
What's wrong with this module, and what error will it raise when you try to
instantiate it?
```python
class MyModel(nn.Module):
    def __init__(self):
        self.linear = nn.Linear(10, 1)
    def forward(self, x):
        return self.linear(x)
```

In [41]:
# Your attempt for Exercise 1 here\n

**Solution 1**

In [42]:
# Solution 1
# Bug: super().__init__() was never called. nn.Module's __init__ sets up
# internal dictionaries (_parameters, _modules, etc.) that assigning a
# submodule (self.linear = nn.Linear(...)) depends on. Without it, Python's
# normal __setattr__ can't hook into nn.Module's special registration logic.
# Error raised: AttributeError: cannot assign module before Module.__init__() call

import torch.nn as nn

class MyModelFixed(nn.Module):
    def __init__(self):
        super().__init__()  # the fix
        self.linear = nn.Linear(10, 1)
    def forward(self, x):
        return self.linear(x)

m = MyModelFixed()
print("works now:", m)


works now: MyModelFixed(
  (linear): Linear(in_features=10, out_features=1, bias=True)
)


**Exercise 2 - Count the parameters.**
For `nn.Linear(10, 5)`, how many total learnable scalars are there (weight +
bias)? Write it out by hand first, then verify with
`sum(p.numel() for p in ...)`.

In [43]:
# Your attempt for Exercise 2 here\n

**Solution 2**

In [44]:
# Solution 2
import torch.nn as nn
# By hand: weight is (out_features, in_features) = (5, 10) = 50 scalars
# bias is (out_features,) = (5,) = 5 scalars
# total = 55

layer = nn.Linear(10, 5)
total = sum(p.numel() for p in layer.parameters())
print("total parameters:", total, "(expected 55)")


total parameters: 55 (expected 55)


**Exercise 3 - Fix the ModuleList bug.**
Rewrite `BadMultiLayer` from section 5 so its parameters are properly
registered, without changing the constructor signature
`__init__(self, sizes)`. Verify the parameter count matches the ModuleList
version.

In [45]:
# Your attempt for Exercise 3 here\n

**Solution 3**

In [46]:
# Solution 3
import torch
import torch.nn as nn

class FixedMultiLayer(nn.Module):
    def __init__(self, sizes):
        super().__init__()
        # the fix: wrap the list in nn.ModuleList instead of a plain list
        self.layers = nn.ModuleList(
            [nn.Linear(sizes[i], sizes[i+1]) for i in range(len(sizes)-1)]
        )

    def forward(self, x):
        for layer in self.layers:
            x = torch.relu(layer(x))
        return x

fixed_model = FixedMultiLayer([10, 20, 20, 1])
print("params found:", len(list(fixed_model.parameters())), "tensors (should be 6)")


params found: 6 tensors (should be 6)


**Exercise 4 - Buffers for real state.**
Write a module `SampleCounter` that tracks the total number of samples it
has ever seen across calls to `forward`, stored as a **buffer** (not a
Python int), so it correctly saves/loads/moves-to-device with the rest of
the model. Prove it appears in `state_dict()` but not `.parameters()`.

In [47]:
# Your attempt for Exercise 4 here\n

**Solution 4**

In [48]:
# Solution 4
import torch
import torch.nn as nn

class SampleCounter(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("total_samples", torch.zeros(1, dtype=torch.long))

    def forward(self, x):
        self.total_samples += x.shape[0]  # add batch size to running total
        return x  # pass-through, this module is just for counting

counter = SampleCounter()
for batch_size in [4, 8, 16]:
    counter(torch.randn(batch_size, 3))

print("total samples seen:", counter.total_samples.item())
print("in state_dict:", counter.state_dict())
print("in parameters (should be empty):", list(counter.parameters()))


total samples seen: 28
in state_dict: OrderedDict({'total_samples': tensor([28])})
in parameters (should be empty): []


**Exercise 5 - train/eval with BatchNorm.**
Create a tiny model with `nn.BatchNorm1d`, run one batch through it in train
mode and the same batch in eval mode, and print how the output differs.
Explain in a comment why they differ.

In [49]:
# Your attempt for Exercise 5 here\n

**Solution 5**

In [50]:
# Solution 5
import torch
import torch.nn as nn

class BNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bn = nn.BatchNorm1d(4)

    def forward(self, x):
        return self.bn(x)

m = BNModel()
x = torch.tensor([[1.0, 2.0, 3.0, 4.0], [5.0, 6.0, 7.0, 8.0]])

m.train()
train_out = m(x)
print("train mode:\n", train_out)

m.eval()
eval_out = m(x)
print("eval mode:\n", eval_out)

# They differ because in train() mode, BatchNorm normalizes using the
# CURRENT batch's mean/variance (computed from just these 2 rows), giving
# a perfectly standardized output. In eval() mode it instead uses the
# RUNNING mean/variance - a moving average that gets updated a little bit
# every time you call the module in train() mode (controlled by the
# `momentum` argument, default 0.1). Since we only ran ONE train-mode batch,
# the running stats shifted only slightly away from their initial values
# (mean=0, var=1), which is why eval_out is close to but not equal to the
# raw input `x` - it's `x` normalized using those slightly-shifted running stats.


train mode:
 tensor([[-1.0000, -1.0000, -1.0000, -1.0000],
        [ 1.0000,  1.0000,  1.0000,  1.0000]],
       grad_fn=<NativeBatchNormBackward0>)
eval mode:
 tensor([[0.5369, 1.2271, 1.9174, 2.6077],
        [3.6047, 4.2950, 4.9853, 5.6755]], grad_fn=<NativeBatchNormBackward0>)


**Exercise 6 - Partial state_dict loading.**
You fine-tuned a model and added one new final layer that didn't exist in
the original checkpoint. Write the `load_state_dict` call that loads the
matching weights and leaves the new layer at its random initialization,
without raising an error.

In [51]:
# Your attempt for Exercise 6 here\n

**Solution 6**

In [52]:
# Solution 6
import torch
import torch.nn as nn

class OldModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 10)

class NewModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 10)
        self.layer2 = nn.Linear(10, 1)  # new layer, not in the old checkpoint

old_model = OldModel()
torch.save(old_model.state_dict(), "/tmp/old_weights.pt")

new_model = NewModel()
# strict=False allows missing/extra keys instead of raising an error
missing_and_unexpected = new_model.load_state_dict(
    torch.load("/tmp/old_weights.pt"), strict=False
)
print(missing_and_unexpected)
print("\nlayer1 loaded from checkpoint, layer2 stayed randomly initialized")


_IncompatibleKeys(missing_keys=['layer2.weight', 'layer2.bias'], unexpected_keys=[])

layer1 loaded from checkpoint, layer2 stayed randomly initialized


**Exercise 7 - Freeze all but the last layer.**
Given a 3-layer `nn.Sequential` model, write code that freezes every layer
except the last one, then verify with `named_parameters()` that only the
last layer's parameters have `requires_grad=True`.

In [53]:
# Your attempt for Exercise 7 here\n

**Solution 7**

In [54]:
# Solution 7
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 1),
)

# freeze everything first
for param in model.parameters():
    param.requires_grad = False

# unfreeze only the last layer (index 2 in this Sequential)
for param in model[2].parameters():
    param.requires_grad = True

for name, param in model.named_parameters():
    print(name, "requires_grad:", param.requires_grad)


0.weight requires_grad: False
0.bias requires_grad: False
2.weight requires_grad: True
2.bias requires_grad: True


**Exercise 8 - Custom learnable layer.**
Write a custom `nn.Module` called `Polynomial` that learns
`y = a*x^2 + b*x + c` where `a`, `b`, `c` are learnable scalar parameters.
Run one manual gradient step and print the gradients for all three.

In [55]:
# Your attempt for Exercise 8 here\n

**Solution 8**

In [56]:
# Solution 8
import torch
import torch.nn as nn

class Polynomial(nn.Module):
    def __init__(self):
        super().__init__()
        self.a = nn.Parameter(torch.tensor(1.0))
        self.b = nn.Parameter(torch.tensor(1.0))
        self.c = nn.Parameter(torch.tensor(1.0))

    def forward(self, x):
        return self.a * x**2 + self.b * x + self.c

model = Polynomial()
x = torch.tensor([1.0, 2.0, 3.0])
target = torch.tensor([3.0, 8.0, 15.0])  # matches y = x^2 + x + 1 roughly

loss = ((model(x) - target) ** 2).mean()
loss.backward()

print("grad a:", model.a.grad.item())
print("grad b:", model.b.grad.item())
print("grad c:", model.c.grad.item())


grad a: -14.666666984558105
grad b: -5.333333492279053
grad c: -2.0


**Exercise 9 - Save and load with a device change.**
Simulate saving a model's `state_dict()` "on GPU" (you can just imagine it,
or actually use GPU if available) and loading it back into a fresh model
instance intended for CPU. What extra argument does `torch.load()` need to
handle this safely, and why?

In [57]:
# Your attempt for Exercise 9 here\n

**Solution 9**

In [58]:
# Solution 9
import torch
import torch.nn as nn

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 2)

gpu_model = TinyModel()
torch.save(gpu_model.state_dict(), "/tmp/model_weights.pt")

cpu_model = TinyModel()
# map_location tells torch.load which device to put the tensors on,
# regardless of which device they were saved from. Without this, loading
# GPU-saved weights on a machine with no GPU raises an error.
state_dict = torch.load("/tmp/model_weights.pt", map_location=torch.device("cpu"))
cpu_model.load_state_dict(state_dict)
print("loaded successfully onto CPU regardless of save-time device")
print("weight device:", cpu_model.linear.weight.device)


loaded successfully onto CPU regardless of save-time device
weight device: cpu


**Exercise 10 - Inspect an unfamiliar model.**
Given the `SmallCNN` class from section 10, write code that prints only the
names and parameter counts of layers that have more than 100 parameters
(i.e. filter out tiny layers from the printout).

In [59]:
# Your attempt for Exercise 10 here\n

**Solution 10**

In [60]:
# Solution 10
import torch
import torch.nn as nn

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3)
        self.fc = nn.Linear(16, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = x.mean(dim=[2, 3])
        return self.fc(x)

model = SmallCNN()
for name, module in model.named_children():
    count = sum(p.numel() for p in module.parameters())
    if count > 100:
        print(f"{name}: {count} parameters")


conv1: 224 parameters
conv2: 1168 parameters
fc: 170 parameters
